# Подготовка датасета: отель

Таблица доступности — ночь проживания и признак, свободен ли номер. Вопросы спрашивают про заезд на несколько ночей: ответ «да», только если свободна каждая ночь периода. 

Справочные вопросы собираются по текстам каталога: номера, специальные предложения и контакты.

Справочные вопросы генерирует локальная модель Ollama `qwen3.8:27b`.

In [ ]:
import json
import random
from datetime import date, timedelta
from pathlib import Path

from tqdm import tqdm

from universal_consultant import generate_availability, generate_client_questions


def project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Не найден корень проекта.")


DATA = project_root() / "data" / "hotels"
START = date(2026, 11, 16)
DAYS = 14
KNOWLEDGE_COUNT = 3
AVAILABILITY_LIMIT = 4
SEED = 7
knowledge: list[dict[str, str]] = []
MAX_NIGHTS = 5
STAY_TEMPLATES = (
    "Можно ли заехать в {subject} {check_in} на {nights}?",
    "Можно ли забронировать {subject} с {check_in} по {check_out}?",
    "Свободен ли {subject} на {nights} с {check_in}?",
)
_MONTHS = (
    "января",
    "февраля",
    "марта",
    "апреля",
    "мая",
    "июня",
    "июля",
    "августа",
    "сентября",
    "октября",
    "ноября",
    "декабря",
)


def _format_date(value: date) -> str:
    return f"{value.day} {_MONTHS[value.month - 1]} {value.year} года"


def _nights_phrase(nights: int) -> str:
    last_two, last = nights % 100, nights % 10
    if last == 1 and last_two != 11:
        word = "ночь"
    elif last in (2, 3, 4) and last_two not in (12, 13, 14):
        word = "ночи"
    else:
        word = "ночей"
    return f"{nights} {word}"


def _sample(items: list, limit: int | None, seed: int) -> list:
    if limit is None or limit >= len(items):
        return list(items)
    if limit < 1:
        raise ValueError("limit должен быть не меньше 1.")
    free = [item for item in items if item["available"]]
    busy = [item for item in items if not item["available"]]
    rng = random.Random(seed)
    rng.shuffle(free)
    rng.shuffle(busy)
    free_count = min(len(free), limit // 2)
    busy_count = min(len(busy), limit - free_count)
    if free_count + busy_count < limit and len(free) > free_count:
        free_count += min(limit - free_count - busy_count, len(free) - free_count)
    elif free_count + busy_count < limit and len(busy) > busy_count:
        busy_count += min(limit - free_count - busy_count, len(busy) - busy_count)
    picked = free[:free_count] + busy[:busy_count]
    rng.shuffle(picked)
    return picked


def stay_questions(
    slots,
    templates,
    *,
    subject: str,
    limit: int | None = None,
    seed: int = 0,
    max_nights: int = 5,
) -> list[dict]:
    """Вопросы про заезд на несколько ночей.

    Ответ «Да», только если свободна каждая ночь с даты заезда до даты выезда, не включая выезд.
    """
    if max_nights < 1:
        raise ValueError("max_nights должен быть не меньше 1.")
    patterns = [template.strip() for template in templates if template.strip()]
    if not patterns:
        raise ValueError("Нужен хотя бы один шаблон вопроса.")
    by_date = {slot.date: slot.available for slot in slots}
    stays = []
    for check_in in sorted(by_date):
        for nights in range(1, max_nights + 1):
            period = [check_in + timedelta(days=offset) for offset in range(nights)]
            if any(day not in by_date for day in period):
                break
            stays.append(
                {
                    "check_in": check_in,
                    "check_out": check_in + timedelta(days=nights),
                    "nights": nights,
                    "available": all(by_date[day] for day in period),
                }
            )
    questions = []
    for index, stay in enumerate(_sample(stays, limit, seed)):
        check_in = _format_date(stay["check_in"])
        check_out = _format_date(stay["check_out"])
        nights = _nights_phrase(stay["nights"])
        questions.append(
            {
                "subject": subject,
                "question": patterns[index % len(patterns)].format(
                    subject=subject,
                    check_in=check_in,
                    check_out=check_out,
                    nights=nights,
                ),
                "check_in": stay["check_in"].isoformat(),
                "check_out": stay["check_out"].isoformat(),
                "nights": stay["nights"],
                "available": stay["available"],
                "answer": "Да" if stay["available"] else "Нет",
            }
        )
    return questions


def read_json(name: str) -> list[dict[str, str]]:
    return json.loads((DATA / name).read_text(encoding="utf-8"))

In [2]:
rooms = read_json("rooms.json")
availability = []
booking = []

for index, room in enumerate(rooms):
    slots = generate_availability(start=START, days=DAYS, seed=SEED + index)
    availability.append(
        {
            "subject": room["name"],
            "slots": [
                {"date": slot.date.isoformat(), "available": slot.available} for slot in slots
            ],
        }
    )
    booking.extend(
        {"kind": "availability", **item}
        for item in stay_questions(
            slots,
            STAY_TEMPLATES,
            subject=room["name"],
            limit=AVAILABILITY_LIMIT,
            seed=SEED + index,
            max_nights=MAX_NIGHTS,
        )
    )

len(availability), len(booking)

(14, 56)

In [11]:
sample = availability[0]
print(sample["subject"])
print(f"{'date':<12} available")
for row in sample["slots"]:
    print(f"{row['date']:<12} {row['available']}")
print()
for item in booking[:6]:
    print(item["answer"], item["nights"], item["question"])

Номер «ДЕЛЮКС»
date         available
2026-11-16   True
2026-11-17   True
2026-11-18   False
2026-11-19   True
2026-11-20   False
2026-11-21   True
2026-11-22   True
2026-11-23   False
2026-11-24   True
2026-11-25   True
2026-11-26   True
2026-11-27   True
2026-11-28   True
2026-11-29   False

Нет 5 Можно ли заехать в Номер «ДЕЛЮКС» 21 ноября 2026 года на 5 ночей?
Да 2 Можно ли забронировать Номер «ДЕЛЮКС» с 21 ноября 2026 года по 23 ноября 2026 года?
Да 2 Свободен ли Номер «ДЕЛЮКС» на 2 ночи с 26 ноября 2026 года?
Нет 4 Можно ли заехать в Номер «ДЕЛЮКС» 18 ноября 2026 года на 4 ночи?
Да 2 Можно ли заехать в Номер «ПРЕМЬЕР» 27 ноября 2026 года на 2 ночи?
Нет 2 Можно ли забронировать Номер «ПРЕМЬЕР» с 18 ноября 2026 года по 20 ноября 2026 года?


In [8]:
(DATA / "availability.json").write_text(
    json.dumps(availability, ensure_ascii=False, indent=4) + "\n", encoding="utf-8"
)

21095

In [10]:
knowledge = []
for doc, use_anchor in [("rooms.json", True), ("special.json", False), ("contacts.json", False)]:
    documents = read_json(doc)

    for document in tqdm(documents):
        try:
            questions = generate_client_questions(
                document["fulltext"],
                subject=document["name"] if use_anchor else None,
                count=KNOWLEDGE_COUNT,
            )
        except RuntimeError as error:
            print(error)
            continue
        knowledge.extend(
            {
                "kind": "knowledge",
                "source": document["name"] + "\n" + document["description"],
                "question": item.question,
                "answer": item.answer,
                "document": document["fulltext"],
            }
            for item in questions
        )

len(knowledge)

100%|██████████| 1/1 [00:12<00:00, 12.32s/it]


72

In [12]:
dataset = [
    {
        "kind": item["kind"],
        "question": item["question"],
        "answer": item["answer"],
    }
    for item in knowledge + booking
]
(DATA / "dataset.json").write_text(
    json.dumps(dataset, ensure_ascii=False, indent=4) + "\n", encoding="utf-8"
)

24913